In [2]:
import seaborn as sns
import csv
import time
import requests
from bs4 import BeautifulSoup

In [ ]:
def normalizar_fecha_escrita(fecha_raw):
    return fecha_raw

urls = {
    "Vitoria": "https://www.booking.com/reviews/es/hotel/libere-vitoria-centro.es.html",
    "Donosti": "https://www.booking.com/reviews/es/hotel/koisi-hostel.es.html",
    "BilbaoMuseo": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-bilbao-guggenheim.es.html",
    "BilbaoLaVieja": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-bilbao-la-vieja.es.html",
    "ValenciaAbastos": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-abastos.es.html",
    "PamplonaYamaguchi": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-pamplona-yamaguchi.es.html",
    "ValenciaJardinBotanico": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-jardin-botanico.es.html",
    "MadridPalacioReal": "https://www.booking.com/reviews/es/hotel/libere-madrid-palacio-real.es.html",
    "MalagaTeatroRomano": "https://www.booking.com/reviews/es/hotel/apartamentosliberemalagateatroromano.es.html",
    "GranadaCatedral": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-granada-catedral.es.html",
    "MalagaLaMerced": "https://www.booking.com/reviews/es/hotel/libere-malaga-la-merced.es.html",
    "CordobaPatio": "https://www.booking.com/reviews/es/hotel/libere-cordoba-patio-santa-marta.es.html"}

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "es-ES,es;q=0.9"}

with open("Datos/Transformados/comentarios.csv", mode="w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)

    writer.writerow([
        "ubicacion",
        "fecha",
        "puntuacion",
        "titulo_comentario",
        "comentario_negativo",
        "comentario_positivo",
        "etiquetas",
        "cantidad_comentarios",
        "nacionalidad"])

    for ubicacion, base_url in urls.items():
        print(f"\nScrapeando {ubicacion}")

        page = 1

        while True:
            print(f"  Página {page}")

            url = f"{base_url}?page={page}"
            response = requests.get(url, headers=headers)
            soup = BeautifulSoup(response.text, "html.parser")

            contenedor = soup.select("li.review_item.clearfix")

            if not contenedor:
                print("No hay mas")
                break

            for reseña in contenedor:

                fecha_el = reseña.select_one("p.review_item_date")
                fecha_raw = fecha_el.get_text(strip=True) if fecha_el else ""
                fecha = normalizar_fecha_escrita(fecha_raw)

                score = reseña.select_one("span.review-score-badge")
                puntuacion = score.get_text(strip=True) if score else ""

                titulo_el = reseña.select_one("span[itemprop='name']")
                titulo = titulo_el.get_text(strip=True) if titulo_el else ""

                neg = reseña.select_one("p.review_neg span[itemprop='reviewBody']")
                comentario_negativo = neg.get_text(strip=True) if neg else ""

                pos = reseña.select_one("p.review_pos span[itemprop='reviewBody']")
                comentario_positivo = pos.get_text(strip=True) if pos else ""

                etiquetas = reseña.select("ul.review_item_info_tags li")
                etiquetas_texto = " | ".join(
                    e.get_text(strip=True).replace("•", "").strip()
                    for e in etiquetas)

                com = reseña.select_one("div.review_item_user_review_count")
                comentarios_n = com.get_text(strip=True) if neg else ""

                nac = reseña.select_one("div.review_item_reviewer span[itemprop='nationality']")
                nacionalidad = nac.get_text(strip=True) if neg else ""

                writer.writerow([
                    ubicacion,
                    fecha,
                    puntuacion,
                    titulo,
                    comentario_negativo,
                    comentario_positivo,
                    etiquetas_texto,
                    comentarios_n,
                    nacionalidad
                ])
            page += 1
            time.sleep(1)


Scrapeando Vitoria
  Página 1
  Página 2
  Página 3
  Página 4
  Página 5
  Página 6
  Página 7
  Página 8
  Página 9
  Página 10
  Página 11
  Página 12
  Página 13
  Página 14
  Página 15
  Página 16
  Página 17
  Página 18
  Página 19
  Página 20
  Página 21
  Página 22
  Página 23
  Página 24
  Página 25
  Página 26
  Página 27
  Página 28
  Página 29
  Página 30
  Página 31
  Página 32
  Página 33
  Página 34
  Página 35
  Página 36
  Página 37
  Página 38
  Página 39
  Página 40
  Página 41
  Página 42
  Página 43
No hay mas

Scrapeando Donosti
  Página 1
  Página 2
  Página 3
  Página 4
  Página 5
  Página 6
  Página 7
  Página 8
  Página 9
  Página 10
  Página 11
  Página 12
  Página 13
  Página 14
  Página 15
  Página 16
  Página 17
  Página 18
No hay mas

Scrapeando BilbaoMuseo
  Página 1
  Página 2
  Página 3
  Página 4
  Página 5
No hay mas

Scrapeando BilbaoLaVieja
  Página 1
  Página 2
  Página 3
  Página 4
  Página 5
  Página 6
  Página 7
No hay mas

Scrapeando Valencia

In [ ]:
import pandas as pd
import re
import os
import matplotlib.pyplot as plt

from collections import Counter
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords

os.makedirs("graficos_wordcloud", exist_ok=True)

df = pd.read_csv("Datos/Transformados/comentarios.csv", encoding="utf-8")

df["comentario_positivo"] = df["comentario_positivo"].fillna("")
df["comentario_negativo"] = df["comentario_negativo"].fillna("")

stop_words = set(stopwords.words("spanish"))


def limpiar_texto(texto):
    texto = texto.lower()
    palabras = re.findall(r'\b\w+\b', texto)
    palabras_filtradas = [
        p for p in palabras
        if p not in stop_words and len(p) > 2
    ]
    return palabras_filtradas


hoteles = df["ubicacion"].unique()

for hotel in hoteles:

    print(f"Procesando {hotel}")

    df_hotel = df[df["ubicacion"] == hotel]

    #comentarios positivos

    texto_pos = " ".join(df_hotel["comentario_positivo"])
    palabras_pos = limpiar_texto(texto_pos)
    freq_pos = Counter(palabras_pos)

    if freq_pos:
        wordcloud_pos = WordCloud(
            width=800,
            height=400,
            background_color="white"
        ).generate_from_frequencies(freq_pos)

        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud_pos, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"{hotel} - Comentarios Positivos")
        plt.tight_layout()
        plt.savefig(f"graficos_wordcloud/{hotel}_wordcloud_positivos.png")
        plt.close()

    #comentarios negativos

    texto_neg = " ".join(df_hotel["comentario_negativo"])
    palabras_neg = limpiar_texto(texto_neg)
    freq_neg = Counter(palabras_neg)

    if freq_neg:
        wordcloud_neg = WordCloud(
            width=800,
            height=400,
            background_color="white"
        ).generate_from_frequencies(freq_neg)

        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud_neg, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"{hotel} - Comentarios Negativos")
        plt.tight_layout()
        plt.savefig(f"graficos_wordcloud/{hotel}_wordcloud_negativos.png")
        plt.close()

Procesando Vitoria
Procesando Donosti
Procesando BilbaoMuseo
Procesando BilbaoLaVieja
Procesando ValenciaAbastos
Procesando PamplonaYamaguchi
Procesando ValenciaJardinBotanico
Procesando MadridPalacioReal
Procesando MalagaTeatroRomano
Procesando GranadaCatedral
Procesando MalagaLaMerced
Procesando CordobaPatio


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs("graficos_prioridad", exist_ok=True)

df = pd.read_csv("Datos/Transformados/comentarios.csv", encoding="utf-8")

df["puntuacion"] = (
    df["puntuacion"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df["comentario_negativo"] = df["comentario_negativo"].fillna("")

problemas_clave = {
    "Limpieza": ["sucio", "sucia", "limpieza", "limpio"],
    "Ruido": ["ruido", "ruidoso", "ruidos"],
    "Cama / Colchón": ["cama", "colchón", "almohada"],
    "Temperatura": ["frío", "fria", "calor", "aire"],
    "Instalaciones": ["baño", "ducha", "habitacion", "habitaciones"],
    "Precio": ["caro", "precio", "coste"],
    "Personal": ["personal", "trato", "recepción"]
}

def contiene_problema(texto, palabras):
    texto = texto.lower()
    return any(p in texto for p in palabras)

def generar_matriz(df_base, titulo, nombre_archivo):

    resultados = []

    for problema, palabras in problemas_clave.items():

        mask = df_base["comentario_negativo"].apply(
            lambda x: contiene_problema(x, palabras)
        )

        df_con = df_base[mask]
        df_sin = df_base[~mask]

        if len(df_con) < 5 or len(df_sin) < 5:
            continue

        impacto = df_sin["puntuacion"].mean() - df_con["puntuacion"].mean()

        resultados.append({
            "problema": problema,
            "frecuencia": len(df_con),
            "impacto": impacto
        })

    if not resultados:
        return

    df_prioridad = pd.DataFrame(resultados)

    mediana_frec = df_prioridad["frecuencia"].median()
    mediana_imp = df_prioridad["impacto"].median()

    colores = []
    for _, row in df_prioridad.iterrows():
        if row["frecuencia"] >= mediana_frec and row["impacto"] >= mediana_imp:
            colores.append("lightcoral")
        elif row["frecuencia"] < mediana_frec and row["impacto"] >= mediana_imp:
            colores.append("wheat")
        elif row["frecuencia"] >= mediana_frec and row["impacto"] < mediana_imp:
            colores.append("lightblue")
        else:
            colores.append("lightgreen")


    plt.figure(figsize=(14, 10))

    plt.scatter(
        df_prioridad["frecuencia"],
        df_prioridad["impacto"],
        s=df_prioridad["frecuencia"] * 25,
        c=colores,
        alpha=0.7,
        edgecolors="black"
    )

    for _, row in df_prioridad.iterrows():
        plt.annotate(
            row["problema"],
            (row["frecuencia"], row["impacto"]),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7)
        )

    y_min = df_prioridad["impacto"].min()
    y_max = df_prioridad["impacto"].max()
    margen_y = (y_max - y_min) * 0.25 if y_max != y_min else 0.5
    plt.ylim(y_min - margen_y, y_max + margen_y)

    plt.axvline(mediana_frec, linestyle="--", alpha=0.5)
    plt.axhline(mediana_imp, linestyle="--", alpha=0.5)

    plt.xlabel("Frecuencia (número de menciones)")
    plt.ylabel("Impacto en la puntuación")
    plt.title(titulo)
    plt.grid(True, alpha=0.3)

    x_min, x_max = plt.xlim()
    y_min, y_max = plt.ylim()

    plt.text(
        x_max * 0.97, y_max * 0.95,
        "Alta prioridad\nResolver primero",
        ha="right", va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="lightcoral", alpha=0.3)
    )

    plt.text(
        x_min + (x_max - x_min) * 0.02, y_max * 0.95,
        "Impacto alto\nPoco frecuente\nInvestigar",
        ha="left", va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="wheat", alpha=0.3)
    )

    plt.text(
        x_max * 0.97, y_min + (y_max - y_min) * 0.02,
        "Frecuente\nBajo impacto\nMejoras menores",
        ha="right", va="bottom",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="lightblue", alpha=0.3)
    )

    plt.text(
        x_min + (x_max - x_min) * 0.02, y_min + (y_max - y_min) * 0.02,
        "Baja prioridad\nIgnorar / Posponer",
        ha="left", va="bottom",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="lightgreen", alpha=0.3)
    )

    nota = (
        "Frecuencia: nº de comentarios donde aparece el problema\n"
        "Impacto: diferencia de puntuación media\n"
        "(Impacto alto = el problema reduce la valoración)"
    )

    plt.figtext(
        0.01, 0.01, nota,
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="whitesmoke", alpha=0.8)
    )

    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    plt.savefig(f"graficos_prioridad/{nombre_archivo}")
    plt.close()

print("\nProcesando análisis GLOBAL")
generar_matriz(
    df,
    "Matriz de Priorización de Problemas - GLOBAL",
    "GLOBAL_matriz_prioridad.png"
)

for hotel in df["ubicacion"].unique():
    print(f"Procesando hotel: {hotel}")
    df_hotel = df[df["ubicacion"] == hotel]
    generar_matriz(
        df_hotel,
        f"Matriz de Priorización de Problemas - {hotel}",
        f"{hotel}_matriz_prioridad.png"
    )



Procesando análisis GLOBAL
Procesando hotel: Vitoria
Procesando hotel: Donosti
Procesando hotel: BilbaoMuseo
Procesando hotel: BilbaoLaVieja
Procesando hotel: ValenciaAbastos
Procesando hotel: PamplonaYamaguchi
Procesando hotel: ValenciaJardinBotanico
Procesando hotel: MadridPalacioReal
Procesando hotel: MalagaTeatroRomano
Procesando hotel: GranadaCatedral
Procesando hotel: MalagaLaMerced
Procesando hotel: CordobaPatio


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import re


os.makedirs("Graficos", exist_ok=True)
os.makedirs("Graficos/evolucion_mensual", exist_ok=True)

df = pd.read_csv("Datos/Transformados/comentarios.csv", encoding="utf-8")

df["puntuacion"] = (
    df["puntuacion"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)
df["puntuacion"] = pd.to_numeric(df["puntuacion"], errors="coerce")
df = df[(df["puntuacion"] >= 5) & (df["puntuacion"] <= 10)]

MESES = {
    "enero": "01", "febrero": "02", "marzo": "03",
    "abril": "04", "mayo": "05", "junio": "06",
    "julio": "07", "agosto": "08",
    "septiembre": "09", "octubre": "10",
    "noviembre": "11", "diciembre": "12"
}

def convertir_fecha(fecha):
    if pd.isna(fecha):
        return None
    fecha = str(fecha).lower().replace("escrito en ", "")
    m = re.search(r"(\d{1,2}) de (\w+) de (\d{4})", fecha)
    if not m:
        return None
    dia, mes_txt, anio = m.groups()
    if mes_txt not in MESES:
        return None
    return pd.to_datetime(f"{anio}-{MESES[mes_txt]}-{dia}")

df["fecha_dt"] = df["fecha"].apply(convertir_fecha)
df = df.dropna(subset=["fecha_dt", "puntuacion"])

df["mes"] = df["fecha_dt"].dt.to_period("M").dt.to_timestamp()

for hotel in df["ubicacion"].unique():

    df_h = df[df["ubicacion"] == hotel]

    resumen = (
        df_h.groupby("mes")
        .agg(media=("puntuacion", "mean"), n=("puntuacion", "count"))
        .sort_index()
    )

    resumen = resumen[resumen["n"] >= 3]
    if len(resumen) < 4:
        continue

    evolucion = resumen["media"]
    trimestral = evolucion.resample("Q").mean()

    mayor_subida = None
    mayor_bajada = None

    for i in range(3, len(evolucion)):
        media_prev = evolucion.iloc[i-3:i].mean()
        actual = evolucion.iloc[i]
        diff = actual - media_prev

        if diff > 0:
            if mayor_subida is None or diff > mayor_subida[2]:
                mayor_subida = (evolucion.index[i], actual, diff)
        else:
            if mayor_bajada is None or diff < mayor_bajada[2]:
                mayor_bajada = (evolucion.index[i], actual, diff)

    plt.figure(figsize=(11, 5))

    plt.plot(
        evolucion.index,
        evolucion.values,
        marker="o",
        linewidth=2,
        label="Media mensual"
    )

    plt.plot(
        trimestral.index,
        trimestral.values,
        linestyle="--",
        color="red",
        linewidth=2,
        label="Media trimestral"
    )

    if mayor_subida:
        mes, valor, diff = mayor_subida
        plt.scatter(mes, valor, s=180, marker="^", color="green")
        plt.annotate(
            f"Subida +{diff:.2f}",
            (mes, valor),
            xytext=(10, 15),
            textcoords="offset points",
            arrowprops=dict(arrowstyle="->"),
            fontsize=9
        )

    if mayor_bajada:
        mes, valor, diff = mayor_bajada
        plt.scatter(mes, valor, s=180, marker="X", color="red")
        plt.annotate(
            f"Bajada {diff:.2f}",
            (mes, valor),
            xytext=(10, -30),
            textcoords="offset points",
            arrowprops=dict(arrowstyle="->"),
            fontsize=9
        )

    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.xticks(rotation=45)

    y_min, y_max = evolucion.min(), evolucion.max()
    margen = max((y_max - y_min) * 0.25, 0.2)
    plt.ylim(y_min - margen, y_max + margen)

    plt.title(f"Evolución mensual – {hotel}")
    plt.xlabel("Mes")
    plt.ylabel("Puntuación media")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        f"Graficos/evolucion_mensual/{hotel}_evolucion_mensual.png"
    )
    plt.close()

hoteles_objetivo = [
    "Vitoria",
    "Donosti",
    "PamplonaYamaguchi"
]

plt.figure(figsize=(12, 6))

for hotel in hoteles_objetivo:

    df_h = df[df["ubicacion"] == hotel]

    resumen = (
        df_h.groupby("mes")
        .agg(media=("puntuacion", "mean"), n=("puntuacion", "count"))
        .sort_index()
    )

    resumen = resumen[resumen["n"] >= 3]
    if len(resumen) < 4:
        continue

    plt.plot(
        resumen.index,
        resumen["media"],
        marker="o",
        linewidth=2,
        label=hotel
    )

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

plt.title("Evolución mensual de la puntuación\nVitoria · Donosti · Pamplona Yamaguchi")
plt.xlabel("Mes")
plt.ylabel("Puntuación media")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig("Graficos/evolucion_mensual/comparativa_3_hoteles.png")
plt.close()


C:\Users\sebas\AppData\Local\Temp\ipykernel_20644\1484365325.py:61: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  trimestral = evolucion.resample("Q").mean()
C:\Users\sebas\AppData\Local\Temp\ipykernel_20644\1484365325.py:61: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  trimestral = evolucion.resample("Q").mean()
C:\Users\sebas\AppData\Local\Temp\ipykernel_20644\1484365325.py:61: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  trimestral = evolucion.resample("Q").mean()
C:\Users\sebas\AppData\Local\Temp\ipykernel_20644\1484365325.py:61: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  trimestral = evolucion.resample("Q").mean()
C:\Users\sebas\AppData\Local\Temp\ipykernel_20644\1484365325.py:61: FutureWarning: 'Q' is deprecated and will be removed in a future version, please

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs("Graficos", exist_ok=True)

df = pd.read_csv("Datos/Transformados/comentarios.csv", encoding="utf-8")

conteo = (
    df.groupby("ubicacion")
    .size()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 6))

plt.bar(conteo.index, conteo.values)

plt.title("Cantidad de comentarios por hotel")
plt.xlabel("Hotel")
plt.ylabel("Número de comentarios")

plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig("Graficos/comentarios_por_hotel.png")
plt.close()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import re

os.makedirs("Graficos", exist_ok=True)
os.makedirs("Graficos/comentarios_mensual", exist_ok=True)

df = pd.read_csv("Datos/Transformados/comentarios.csv", encoding="utf-8")

MESES = {
    "enero": "01", "febrero": "02", "marzo": "03",
    "abril": "04", "mayo": "05", "junio": "06",
    "julio": "07", "agosto": "08",
    "septiembre": "09", "octubre": "10",
    "noviembre": "11", "diciembre": "12"
}

def convertir_fecha(fecha):
    if pd.isna(fecha):
        return None
    fecha = str(fecha).lower().replace("escrito en ", "")
    m = re.search(r"(\d{1,2}) de (\w+) de (\d{4})", fecha)
    if not m:
        return None
    dia, mes_txt, anio = m.groups()
    if mes_txt not in MESES:
        return None
    return pd.to_datetime(f"{anio}-{MESES[mes_txt]}-{dia}")

df["fecha_dt"] = df["fecha"].apply(convertir_fecha)
df = df.dropna(subset=["fecha_dt"])

df["mes"] = df["fecha_dt"].dt.to_period("M").dt.to_timestamp()

df["positivo"] = df["comentario_positivo"].notna()
df["negativo"] = df["comentario_negativo"].notna()

for hotel in df["ubicacion"].unique():

    df_h = df[df["ubicacion"] == hotel]

    resumen = (
        df_h.groupby("mes")
        .agg(
            positivos=("positivo", "sum"),
            negativos=("negativo", "sum")
        )
        .sort_index()
    )

    if len(resumen) < 2:
        continue

    plt.figure(figsize=(11, 5))

    plt.plot(
        resumen.index,
        resumen["positivos"],
        marker="o",
        linewidth=2,
        color="green",
        label="Comentarios positivos"
    )

    plt.plot(
        resumen.index,
        resumen["negativos"],
        marker="o",
        linewidth=2,
        color="red",
        label="Comentarios negativos"
    )

    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.xticks(rotation=45)

    plt.title(f"Comentarios positivos y negativos por mes – {hotel}")
    plt.xlabel("Mes")
    plt.ylabel("Número de comentarios")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        f"Graficos/comentarios_mensual/{hotel}_positivos_negativos.png"
    )
    plt.close()
